# Bankkruptcy Prediction

## Data loader

In [ ]:
# Calling the data loader and preprocessing script to load the data and perform necessary preprocessing steps for bankruptcy prediction analysis.
%run data_loader_panel.py

Loaded: df_active (Active_companies.xlsx)

DataFrame: df_active

  df_active
  Rows: 6,400    Columns: 115

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |      6,400 |      0 |
|   1 | Inactive                          | str     |      6,400 |      0 |
|   2 | Quoted                            | str     |      6,400 |      0 |
|   3 | Branch                            | str     |      6,400 |      0 |
|   4 | OwnData                           | str     |      6,400 |      0 |
|   5 | Woco                              | str     |      6,400 |      0 |
|   6 | Country ISO code                  | str     |      6,400 |      0 |
|   7 | NACE Rev. 2, core code (4 digits) | str     |      6,400 |      0 |
|   8 | Consolidation code                | str     |      6,400 |      0 |
|   9 | P/L before tax EUR 2024           | Float64 |   

## Data loading and preprocessing 

In [42]:
# Merge active and bankrupt into one analysis DataFrame
df_active["Bankrupt"]   = 0
df_bankrupt["Bankrupt"] = 1

df_analysis = pd.concat([df_active, df_bankrupt], ignore_index=True)

nace_section_map = {
    range(1, 4):    "A - Agriculture, forestry, fishing",
    range(5, 10):   "B - Mining and quarrying",
    range(10, 34):  "C - Manufacturing",
    range(36, 40):  "E - Water supply, waste management",
    range(41, 44):  "F - Construction",
    range(45, 48):  "G - Wholesale and retail trade",
    range(49, 54):  "H - Transport and storage",
    range(55, 57):  "I - Accommodation and food service",
    range(58, 64):  "J - Information and communication",
    range(68, 69):  "L - Real estate",
    range(69, 76):  "M - Professional, scientific, technical",
    range(77, 83):  "N - Administrative and support services",
    range(90, 94):  "R - Arts, entertainment, recreation",
    range(94, 97):  "S - Other service activities",
}

def nace_to_section(code):
    try:
        division = int(str(code)[:2])
        for r, section in nace_section_map.items():
            if division in r:
                return section
        return "Unknown"
    except:
        return "Unknown"

df_analysis["NACE Section"] = df_analysis["NACE Rev. 2, core code (4 digits)"].apply(nace_to_section)



print(f"df_analysis: {len(df_analysis):,} rows x {len(df_analysis.columns)} columns")
print(f"  Active:   {(df_analysis['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt: {(df_analysis['Bankrupt'] == 1).sum():,}")

# Add company age column (years since Year of incorporation) after 'Inactive'
year_col = "Year of incorporation"
age_col = "Company Age"
current_year = pd.Timestamp.now().year

years = pd.to_numeric(df_analysis[year_col], errors="coerce")
df_analysis.insert(
    df_analysis.columns.get_loc("Inactive") + 1,
    age_col,
    (current_year - years).astype("Int64"),
)

drop_cols = [
    "Inactive",
    "Quoted",
    "Branch",
    "OwnData",
    "Woco",
    "Country ISO code",
    "Consolidation code",
    "NACE Rev. 2, core code (4 digits)",
]

df_analysis = df_analysis.drop(columns=drop_cols)

move_cols = ["Year of incorporation", "Bankrupt", "NACE Section"]
cols = [c for c in df_analysis.columns if c not in move_cols]
insert_at = cols.index("Company Age") + 1

for col in reversed(move_cols):
    cols.insert(insert_at, col)

df_analysis = df_analysis[cols]

print_df_structure("df_analysis", df_analysis)

df_analysis: 7,527 rows x 117 columns
  Active:   6,400
  Bankrupt: 1,127

DataFrame: df_analysis

  df_analysis
  Rows: 7,527    Columns: 110

|   # | Column                       | Dtype   |   Non-Null |   Null |
|-----|------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet  | str     |      7,527 |      0 |
|   1 | Company Age                  | Int64   |      7,527 |      0 |
|   2 | Year of incorporation        | Int16   |      7,527 |      0 |
|   3 | Bankrupt                     | int64   |      7,527 |      0 |
|   4 | NACE Section                 | str     |      7,527 |      0 |
|   5 | P/L before tax EUR 2024      | Float64 |      6,565 |    962 |
|   6 | P/L before tax EUR 2023      | Float64 |      6,816 |    711 |
|   7 | P/L before tax EUR 2022      | Float64 |      6,866 |    661 |
|   8 | P/L before tax EUR 2021      | Float64 |      6,535 |    992 |
|   9 | P/L before tax EUR 2020      | Float64 |      6,110 |  1,417 |
|  1

In [43]:
from IPython.display import display, HTML
import pandas as pd

def display_scrollable(obj, height=400, na_rep=""):
    if isinstance(obj, pd.DataFrame):
        html = obj.to_html(classes="scrollable-table", na_rep=na_rep)
    else:
        html = "<pre>{}</pre>".format(HTML.escape(str(obj)))
    style = f"""
    <style>
      .scrollable-table-wrapper {{ max-height: {height}px; overflow: auto; border:1px solid #ddd; }}
      .scrollable-table {{ border-collapse: collapse; width:100%; }}
    </style>
    """
    display(HTML(style + f'<div class="scrollable-table-wrapper">{html}</div>'))

# Example: show the first 200 rows of df_analysis in a scrollable pane
display_scrollable(df_analysis.head(200), height=500)

,Company name Latin alphabet,Company Age,Year of incorporation,Bankrupt,NACE Section,P/L before tax EUR 2024,P/L before tax EUR 2023,P/L before tax EUR 2022,P/L before tax EUR 2021,P/L before tax EUR 2020,P/L before tax EUR 2019,P/L before tax EUR 2018,P/L before tax EUR 2017,P/L before tax EUR 2016,P/L before tax EUR 2015,P/L before tax EUR 2014,P/L before tax EUR 2013,P/L before tax EUR 2012,P/L before tax EUR 2011,P/L before tax EUR 2010,P/L before tax EUR 2009,P/L before tax EUR 2008,P/L before tax EUR 2007,P/L before tax EUR 2006,P/L before tax EUR 2005,P/L before tax EUR 2004,Total assets EUR 2024,Total assets EUR 2023,Total assets EUR 2022,Total assets EUR 2021,Total assets EUR 2020,Total assets EUR 2019,Total assets EUR 2018,Total assets EUR 2017,Total assets EUR 2016,Total assets EUR 2015,Total assets EUR 2014,Total assets EUR 2013,Total assets EUR 2012,Total assets EUR 2011,Total assets EUR 2010,Total assets EUR 2009,Total assets EUR 2008,Total assets EUR 2007,Total assets EUR 2006,Total assets EUR 2005,Total assets EUR 2004,Total liabilities EUR 2024,Total liabilities EUR 2023,Total liabilities EUR 2022,Total liabilities EUR 2021,Total liabilities EUR 2020,Total liabilities EUR 2019,Total liabilities EUR 2018,Total liabilities EUR 2017,Total liabilities EUR 2016,Total liabilities EUR 2015,Total liabilities EUR 2014,Total liabilities EUR 2013,Total liabilities EUR 2012,Total liabilities EUR 2011,Total liabilities EUR 2010,Total liabilities EUR 2009,Total liabilities EUR 2008,Total liabilities EUR 2007,Total liabilities EUR 2006,Total liabilities EUR 2005,Total liabilities EUR 2004,Current liabilities EUR 2024,Current liabilities EUR 2023,Current liabilities EUR 2022,Current liabilities EUR 2021,Current liabilities EUR 2020,Current liabilities EUR 2019,Current liabilities EUR 2018,Current liabilities EUR 2017,Current liabilities EUR 2016,Current liabilities EUR 2015,Current liabilities EUR 2014,Current liabilities EUR 2013,Current liabilities EUR 2012,Current liabilities EUR 2011,Current liabilities EUR 2010,Current liabilities EUR 2009,Current liabilities EUR 2008,Current liabilities EUR 2007,Current liabilities EUR 2006,Current liabilities EUR 2005,Current liabilities EUR 2004,Current assets EUR 2024,Current assets EUR 2023,Current assets EUR 2022,Current assets EUR 2021,Current assets EUR 2020,Current assets EUR 2019,Current assets EUR 2018,Current assets EUR 2017,Current assets EUR 2016,Current assets EUR 2015,Current assets EUR 2014,Current assets EUR 2013,Current assets EUR 2012,Current assets EUR 2011,Current assets EUR 2010,Current assets EUR 2009,Current assets EUR 2008,Current assets EUR 2007,Current assets EUR 2006,Current assets EUR 2005,Current assets EUR 2004
0,MOBILE-TV OY,29,1997,0,J - Information and communication,-9000.0,40000.0,-7000.0,45000.0,-16000.0,47000.0,-3000.0,90000.0,27000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,269000.0,319000.0,290000.0,317000.0,302000.0,367000.0,357000.0,437000.0,354000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,46000.0,69000.0,55000.0,54000.0,58000.0,85000.0,114000.0,83000.0,54000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,215000.0,237000.0,194000.0,223000.0,196000.0,240000.0,216000.0,312000.0,260000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,MRK KONEURAKOINTI OY,14,2012,0,F - Construction,-28000.0,42000.0,-2000.0,-54000.0,39000.0,110000.0,28000.0,14000.0,23000.0,19000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,136000.0,193000.0,178000.0,232000.0,244000.0,235000.0,153000.0,204000.0,206000.0,193000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,50000.0,50000.0,35000.0,43000.0,32000.0,65000.0,52000.0,59000.0,51000.0,50000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>

In [39]:
# Verify
print(df_analysis.groupby("NACE Section")["Bankrupt"].agg(['count','mean']).round(3))

                                         count   mean
NACE Section                                         
A - Agriculture, forestry, fishing         261  0.080
B - Mining and quarrying                    23  0.043
C - Manufacturing                          790  0.149
E - Water supply, waste management          62  0.081
F - Construction                          1361  0.220
G - Wholesale and retail trade            1407  0.155
H - Transport and storage                  558  0.159
I - Accommodation and food service         463  0.197
J - Information and communication          377  0.082
L - Real estate                            558  0.056
M - Professional, scientific, technical    912  0.087
N - Administrative and support services    498  0.209
R - Arts, entertainment, recreation        172  0.134
S - Other service activities                85  0.188


In [34]:
# Unique consolidation codes
consol_counts = (
    df_analysis.groupby("Consolidation code")["Bankrupt"]
    .agg(Total="count", Active=lambda x: (x == 0).sum(), Bankrupt=lambda x: (x == 1).sum())
    .reset_index()
    .sort_values("Total", ascending=False)
)

print(tabulate(
    consol_counts,
    headers=["Consolidation Code", "Total", "Active", "Bankrupt"],
    tablefmt="github",
    numalign="right",
    showindex=False,
))

| Consolidation Code   |   Total |   Active |   Bankrupt |
|----------------------|---------|----------|------------|
| U1                   |    7037 |     5978 |       1059 |
| U2                   |     301 |      255 |         46 |
| C2                   |     189 |      167 |         22 |
